# SpecReX: in-silico spectroscopy demonstration

This notebook explains a deliberately simple four-class spectral classifier using
Responsibility-based eXplanations (ReX).

It runs from a fresh Google Colab session and automatically downloads:

- 50 simulated spectra from each of four classes;
- the corresponding labels and known ground-truth feature masks;
- the common wavenumber axis; and
- the trained PyTorch model weights.

The complete dataset contains 200 spectra. For a responsive live demonstration, ReX
defaults to five spectra per class. One setting can increase this to all 50.

Repository: [nathanblakekcl/ReX](https://github.com/nathanblakekcl/ReX/tree/feature/spec_tutorial)

## 1. Install the spectroscopy-enabled ReX branch

ReX currently pins an older NumPy version. Colab already provides compatible scientific
packages, so the branch is installed without replacing Colab's NumPy. This avoids the
binary incompatibility caused by changing NumPy inside a running session.

In [ ]:
%pip install -q imutils toml anytree tqdm sqlalchemy onnxruntime tabulate
%pip install -q --no-deps "git+https://github.com/nathanblakekcl/ReX.git@feature/spec_tutorial"

## 2. Imports and compute device

In [ ]:
from pathlib import Path
from urllib.request import urlretrieve

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm.auto import tqdm

from rex_xai.lib import ReX

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using: {device}")

## 3. Download the demonstration data and model

The files are versioned with the code on GitHub; no Drive mount or manual upload is
required.

In [ ]:
branch = "feature/spec_tutorial"
asset_base = (
    "https://raw.githubusercontent.com/nathanblakekcl/ReX/"
    f"{branch}/tests/test_data/insilico_demo"
)
data_dir = Path("specrex_insilico_demo")
data_dir.mkdir(exist_ok=True)

asset_names = [
    "insilico_demo_data.npz",
    "insilico_demo_model.pt",
]

for asset_name in asset_names:
    destination = data_dir / asset_name
    if not destination.exists():
        print(f"Downloading {asset_name}...")
        urlretrieve(f"{asset_base}/{asset_name}", destination)

print(f"Assets ready in: {data_dir.resolve()}")

## 4. Load and validate the balanced dataset

The saved spectra are raw simulated intensities. We apply the same row-wise z-score
normalization used during model training.

In [ ]:
saved = np.load(data_dir / "insilico_demo_data.npz")

spectra = saved["spectra"].astype(np.float32)
labels = saved["labels"].astype(int)
ground_truth_mask = saved["ground_truth_mask"].astype(bool)
wavenumbers = saved["wavenumbers"].astype(float)


def normalize_spectra(values):
    values = np.asarray(values, dtype=np.float32)
    means = values.mean(axis=1, keepdims=True)
    standard_deviations = values.std(axis=1, keepdims=True)
    if np.any(standard_deviations < 1e-12):
        raise ValueError("At least one spectrum has effectively zero variance")
    return (values - means) / standard_deviations


spectra_normalized = normalize_spectra(spectra)
unique_labels, class_counts = np.unique(labels, return_counts=True)

expected_shape = (len(spectra), spectra.shape[1])
if labels.shape != (len(spectra),):
    raise ValueError("There must be one label per spectrum")
if ground_truth_mask.shape != expected_shape:
    raise ValueError(
        f"Ground-truth shape {ground_truth_mask.shape} does not match {expected_shape}"
    )
if wavenumbers.shape != (spectra.shape[1],):
    raise ValueError("Wavenumber axis length does not match the spectra")
if not np.isfinite(spectra_normalized).all():
    raise ValueError("Normalized spectra contain NaN or infinite values")

print("Raw spectra:", spectra.shape)
print("Normalized spectra:", spectra_normalized.shape)
print("Class counts:", dict(zip(unique_labels, class_counts)))
print("Ground-truth masks:", ground_truth_mask.shape)
print("Wavenumbers:", wavenumbers.shape)

## 5. Plot the mean spectrum for every class

Class names are deliberately generic because the classes are synthetic. Replace
`class_names` if you have more descriptive names for the simulated conditions.

In [ ]:
class_names = {
    0: "Class 0",
    1: "Class 1",
    2: "Class 2",
    3: "Class 3",
}

plt.figure(figsize=(11, 6))

for label in unique_labels:
    group_spectra = spectra_normalized[labels == label]
    mean_spectrum = group_spectra.mean(axis=0)
    label_name = class_names.get(int(label), f"Class {label}")

    plt.plot(
        wavenumbers,
        mean_spectrum,
        label=f"{label_name} (n={len(group_spectra)})",
    )

plt.xlabel(r"Wavenumber (cm$^{-1}$)")
plt.ylabel("Mean normalized intensity")
plt.title("Mean simulated spectrum for each class")
plt.legend()
plt.tight_layout()
plt.show()

## 6. Recreate the model and load its trained weights

The architecture must match the model used during training. Its flatten dimension is
calculated from the input length, so the definition is not tied to a hard-coded number
of spectral points.

In [ ]:
class ConvNet(nn.Module):
    def __init__(self, input_length, num_classes=4):
        super().__init__()
        self.layer1 = nn.Sequential(
            nn.Conv1d(1, 20, kernel_size=7, stride=1),
            nn.BatchNorm1d(20),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=7, stride=3),
        )
        self.layer2 = nn.Sequential(
            nn.Conv1d(20, 40, kernel_size=5, stride=1),
            nn.BatchNorm1d(40),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=5, stride=2),
        )
        self.layer3 = nn.Sequential(
            nn.Conv1d(40, 40, kernel_size=3, stride=1),
            nn.BatchNorm1d(40),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=3, stride=1),
        )

        with torch.no_grad():
            dummy = torch.zeros(1, 1, input_length)
            output = self.layer3(self.layer2(self.layer1(dummy)))
            flatten_dimension = output.numel()

        self.fc1 = nn.Linear(flatten_dimension, 60)
        self.fc2 = nn.Linear(60, num_classes)

    def forward(self, x):
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = x.reshape(x.size(0), -1)
        x = F.relu(self.fc1(x))
        x = F.dropout(x, 0.2, training=self.training)
        return self.fc2(x)


model = ConvNet(
    input_length=spectra_normalized.shape[1],
    num_classes=len(unique_labels),
).to(device)

state_dict = torch.load(
    data_dir / "insilico_demo_model.pt",
    map_location=device,
    weights_only=True,
)
model.load_state_dict(state_dict)
model.eval()
print(model.__class__.__name__, "loaded")

## 7. Evaluate predictions and confidence

Confidence is the largest softmax probability. It describes the model's output, not
whether that output is scientifically reliable.

In [ ]:
x_all = torch.as_tensor(
    spectra_normalized,
    dtype=torch.float32,
    device=device,
).unsqueeze(1)

model.eval()
with torch.no_grad():
    logits = model(x_all)
    probabilities = torch.softmax(logits, dim=1)

predicted_labels = probabilities.argmax(dim=1).cpu().numpy()
confidences = probabilities.max(dim=1).values.cpu().numpy()
accuracy = float(np.mean(predicted_labels == labels))

print(f"Accuracy on the 200-spectrum demo subset: {accuracy:.1%}")

confusion = np.zeros(
    (len(unique_labels), len(unique_labels)),
    dtype=int,
)
for true_label, predicted_label in zip(labels, predicted_labels):
    confusion[int(true_label), int(predicted_label)] += 1

fig, ax = plt.subplots(figsize=(6, 5))
image = ax.imshow(confusion, cmap="Blues")
for row in range(confusion.shape[0]):
    for column in range(confusion.shape[1]):
        ax.text(
            column,
            row,
            confusion[row, column],
            ha="center",
            va="center",
        )
ax.set_xticks(range(len(unique_labels)), unique_labels)
ax.set_yticks(range(len(unique_labels)), unique_labels)
ax.set_xlabel("Predicted label")
ax.set_ylabel("Ground-truth label")
ax.set_title("Confusion matrix")
fig.colorbar(image, ax=ax)
fig.tight_layout()
plt.show()

## 8. Configure the live ReX demonstration

The dataset contains 50 spectra per class. Running ReX on all 200 may take a long time,
particularly on a free CPU runtime.

- Keep `SPECTRA_PER_CLASS_TO_EXPLAIN = 5` for a live demonstration.
- Set it to `50` to process the complete balanced dataset.
- Increase `REX_ITERATIONS` for more stable maps at the cost of runtime.

In [ ]:
SPECTRA_PER_CLASS_TO_EXPLAIN = 5
REX_ITERATIONS = 20
REX_MASK_VALUE = "linear"  # "linear" or "quadratic"
RANDOM_SEED = 13

if not 1 <= SPECTRA_PER_CLASS_TO_EXPLAIN <= 50:
    raise ValueError("SPECTRA_PER_CLASS_TO_EXPLAIN must be between 1 and 50")
if REX_MASK_VALUE not in {"linear", "quadratic"}:
    raise ValueError("REX_MASK_VALUE must be 'linear' or 'quadratic'")

selection_rng = np.random.default_rng(RANDOM_SEED)
selected_indices = []

for label in unique_labels:
    class_indices = np.flatnonzero(labels == label)
    selected_indices.extend(
        selection_rng.choice(
            class_indices,
            size=SPECTRA_PER_CLASS_TO_EXPLAIN,
            replace=False,
        )
    )

selected_indices = np.asarray(selected_indices, dtype=int)
print(
    f"Selected {len(selected_indices)} spectra "
    f"({SPECTRA_PER_CLASS_TO_EXPLAIN} per class)"
)

## 9. Calculate ReX responsibility maps

Each map explains the class selected by the model for that spectrum. ReX perturbs
spectral regions and measures which retained positions are responsible for preserving
that prediction.

In [ ]:
def calculate_responsibility_maps(
    model,
    spectra,
    indices,
    device,
    *,
    mask_value="linear",
    iterations=20,
    seed=13,
):
    input_length = spectra.shape[1]
    rex = ReX(
        model,
        ["N", 1, input_length],
        device,
        "spectral",
    )
    rex.args.mask_value = mask_value
    rex.args.iters = int(iterations)
    rex.args.seed = int(seed)
    rex.args.progress_bar = False

    if mask_value == "quadratic":
        rex.args.quad_sigma = 0.0

    responsibility_maps = []
    stats = []

    for index in tqdm(indices, desc="Calculating ReX responsibility"):
        sample = torch.as_tensor(
            spectra[index],
            dtype=torch.float32,
            device=device,
        ).unsqueeze(0)

        rex.set_tabular_data(f"spectrum-{index}", data=sample)
        rex.set_target()
        rex.calculate_responsibility()

        target_class = rex.data.target.classification
        responsibility = rex.maps.get(target_class)

        if responsibility is None:
            raise RuntimeError(
                f"No responsibility map was produced for spectrum {index}; "
                f"inspect ReX statistics: {rex.stats}"
            )

        responsibility = np.asarray(
            responsibility,
            dtype=float,
        ).squeeze()

        if responsibility.shape != (input_length,):
            raise RuntimeError(
                f"Spectrum {index}: map shape {responsibility.shape} "
                f"does not match ({input_length},)"
            )

        responsibility_maps.append(responsibility)
        stats.append(rex.stats)

    return np.stack(responsibility_maps), stats


responsibility_maps, rex_stats = calculate_responsibility_maps(
    model,
    spectra_normalized,
    selected_indices,
    device,
    mask_value=REX_MASK_VALUE,
    iterations=REX_ITERATIONS,
    seed=RANDOM_SEED,
)

print("Responsibility maps:", responsibility_maps.shape)

## 10. Plotting helpers

Red intensity represents normalized ReX responsibility. Green shading marks the known
simulated ground-truth feature regions. The raw ReX values are not calibrated
probabilities.

In [ ]:
def spectral_edges(axis):
    axis = np.asarray(axis, dtype=float)
    if axis.ndim != 1 or len(axis) < 2:
        raise ValueError("axis must contain at least two points")
    differences = np.diff(axis)
    if not np.all(differences > 0) and not np.all(differences < 0):
        raise ValueError("axis must be strictly monotonic")

    edges = np.empty(len(axis) + 1, dtype=float)
    edges[1:-1] = (axis[:-1] + axis[1:]) / 2
    edges[0] = axis[0] - differences[0] / 2
    edges[-1] = axis[-1] + differences[-1] / 2
    return edges


def plot_spectral_explanation(
    axis,
    spectrum,
    responsibility,
    *,
    title,
    known_mask=None,
):
    axis = np.asarray(axis)
    spectrum = np.asarray(spectrum)
    responsibility = np.asarray(responsibility)

    if not (axis.shape == spectrum.shape == responsibility.shape):
        raise ValueError(
            "axis, spectrum and responsibility must have matching shapes"
        )

    maximum = np.nanmax(responsibility)
    normalized = (
        responsibility / maximum
        if np.isfinite(maximum) and maximum > 0
        else np.zeros_like(responsibility)
    )

    minimum_intensity = float(spectrum.min())
    maximum_intensity = float(spectrum.max())

    fig, ax = plt.subplots(figsize=(12, 5))
    ax.pcolormesh(
        spectral_edges(axis),
        [minimum_intensity, maximum_intensity],
        normalized.reshape(1, -1),
        cmap="Reds",
        shading="auto",
        alpha=0.55,
    )

    if known_mask is not None:
        known_mask = np.asarray(known_mask, dtype=bool)
        if known_mask.shape != spectrum.shape:
            raise ValueError("known_mask must match the spectrum shape")
        ax.fill_between(
            axis,
            minimum_intensity,
            maximum_intensity,
            where=known_mask,
            color="green",
            alpha=0.15,
            label="Known simulated feature",
        )

    ax.plot(axis, spectrum, color="black", linewidth=1)
    ax.set_xlabel(r"Wavenumber (cm$^{-1}$)")
    ax.set_ylabel("Normalized intensity")
    ax.set_title(title)
    if known_mask is not None:
        ax.legend(loc="upper right")
    fig.tight_layout()
    plt.show()

## 11. Plot every selected spectrum separately

Each title reports the spectrum index, ground-truth class, model prediction, and model
confidence.

In [ ]:
for map_index, spectrum_index in enumerate(selected_indices):
    true_id = int(labels[spectrum_index])
    predicted_id = int(predicted_labels[spectrum_index])

    true_name = class_names.get(true_id, f"Class {true_id}")
    predicted_name = class_names.get(
        predicted_id,
        f"Class {predicted_id}",
    )

    plot_spectral_explanation(
        wavenumbers,
        spectra_normalized[spectrum_index],
        responsibility_maps[map_index],
        known_mask=ground_truth_mask[spectrum_index],
        title=(
            f"Spectrum {spectrum_index} | "
            f"Ground truth: {true_name} | "
            f"Prediction: {predicted_name} | "
            f"Confidence: {confidences[spectrum_index]:.1%}"
        ),
    )

## 12. Plot mean spectrum and mean responsibility by class

These means use only the spectra selected for ReX in Section 8. Increase
`SPECTRA_PER_CLASS_TO_EXPLAIN` for a more stable class-level map.

In [ ]:
selected_labels = labels[selected_indices]

for label in unique_labels:
    group = selected_labels == label
    group_indices = selected_indices[group]

    mean_spectrum = spectra_normalized[group_indices].mean(axis=0)
    mean_responsibility = responsibility_maps[group].mean(axis=0)
    union_ground_truth = ground_truth_mask[group_indices].any(axis=0)
    label_name = class_names.get(int(label), f"Class {label}")

    plot_spectral_explanation(
        wavenumbers,
        mean_spectrum,
        mean_responsibility,
        known_mask=union_ground_truth,
        title=(
            f"Mean SpecReX explanation — {label_name} "
            f"(n={len(group_indices)})"
        ),
    )

## 13. Locate maximum mean responsibility by class

This is a compact numerical summary of the preceding plots.

In [ ]:
for label in unique_labels:
    group = selected_labels == label
    mean_responsibility = responsibility_maps[group].mean(axis=0)
    peak_index = int(np.nanargmax(mean_responsibility))
    label_name = class_names.get(int(label), f"Class {label}")

    print(
        f"{label_name}: maximum mean responsibility at "
        f"{wavenumbers[peak_index]:.2f} cm⁻¹ "
        f"(index {peak_index})"
    )

# Adapt the workflow to your own data

To explain your own spectra:

1. Load spectra with shape `[samples, spectral_points]`.
2. Apply exactly the preprocessing used to train your classifier.
3. Load a PyTorch model accepting `[batch, 1, spectral_points]`.
4. Replace `spectra_normalized`, `model`, `labels`, and `wavenumbers`.
5. Select indices and call `calculate_responsibility_maps`.

The bundled model is specific to these 1,266-point simulated spectra. It is not a
general Raman classifier.

## Interpretation checks

- Responsibility reveals what the classifier relies upon, not biochemical causality.
- Compare the responsibility overlay with the known simulated mask.
- Repeat with multiple random seeds or more iterations before interpreting fine detail.
- Compare linear and quadratic fill strategies.
- Treat confident but incorrect predictions as model failures, not successful
  explanations.